# Fiche 1 — La maille d'une table : une ligne, c'est quoi ?

| | |
|---|---|
| 🎯 **Objectif** | Dire ce que représente une ligne, vérifier qu'une clé est unique et repérer des doublons créés par une jointure |
| ⏱️ **Durée** | 30 min |
| 🧰 **Prérequis** | pandas : lire un CSV, `groupby` |
| 🔗 **Dans le projet** | Étape 1 (exploration, nettoyage) |

> **L'idée clé.** Avant tout calcul, on se demande : **une ligne = quoi ?** (ici : un pays, une culture, une année). Si cette combinaison apparaît plusieurs fois, chaque statistique, chaque modèle, chaque découpage train/test est faussé.

## 1. La table « prête à l'emploi » du projet

In [1]:
import pandas as pd

from agrireco import io

_, fao_dir = io.download()  # téléchargement vérifié par empreinte SHA-256 (rien à faire s'il est déjà là)

yield_df = pd.read_csv(fao_dir / "yield_df.csv", index_col=0)
print(yield_df.shape)
yield_df.head(3)

(28242, 7)


,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37


On **devine** qu'une ligne = un pays (`Area`), une culture (`Item`), une année (`Year`). On ne devine pas : on vérifie.
`duplicated(subset=...)` marque les lignes dont la combinaison de colonnes est déjà apparue plus haut.

In [2]:
cle = ["Area", "Item", "Year"]
print("lignes                       :", len(yield_df))
print("combinaisons distinctes      :", len(yield_df.drop_duplicates(cle)))
print("lignes en trop (doublons)    :", yield_df.duplicated(cle).sum())

lignes                       : 28242
combinaisons distinctes      : 13130
lignes en trop (doublons)    : 15112


**Plus de la moitié des lignes sont en trop.** Regardons-en une :

In [3]:
yield_df[(yield_df["Area"] == "India") & (yield_df["Item"] == "Wheat") & (yield_df["Year"] == 2000)].head(6)

,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
12416,India,Wheat,2000,27785,1083.0,44957.52,25.98
12417,India,Wheat,2000,27785,1083.0,44957.52,27.44
12418,India,Wheat,2000,27785,1083.0,44957.52,26.13
12419,India,Wheat,2000,27785,1083.0,44957.52,24.57
12420,India,Wheat,2000,27785,1083.0,44957.52,25.34
12421,India,Wheat,2000,27785,1083.0,44957.52,25.93


Même pays, même culture, même année, **même rendement** ; seule la température change. L'Inde a plusieurs stations
météo dans le fichier de températures, et chaque station a produit une ligne lors de la jointure.

## ⚠️ Piège : un doublon change les moyennes (et pas seulement le nombre de lignes)

In [4]:
ble = yield_df[yield_df["Item"] == "Wheat"]
print("rendement moyen du blé, table brute     :", round(ble["hg/ha_yield"].mean()), "hg/ha")
print("rendement moyen du blé, sans doublons   :", round(ble.drop_duplicates(cle)["hg/ha_yield"].mean()), "hg/ha")
print("poids de l'Inde dans la table brute     :", f"{(ble['Area'] == 'India').mean():.1%}")
print("poids de l'Inde sans doublons           :", f"{(ble.drop_duplicates(cle)['Area'] == 'India').mean():.1%}")

rendement moyen du blé, table brute     : 30116 hg/ha
rendement moyen du blé, sans doublons   : 30783 hg/ha
poids de l'Inde dans la table brute     : 13.1%
poids de l'Inde sans doublons           : 1.3%


L'Inde pèse beaucoup plus qu'elle ne le devrait : la moyenne « mondiale » du blé est tirée vers le rendement indien.
Un modèle entraîné sur cette table verrait l'Inde **dix fois** plus qu'elle ne le devrait, et un découpage aléatoire mettrait des
copies de la même ligne à la fois dans l'entraînement et dans le test.

## 2. Remonter à la cause
Les doublons viennent du fichier de températures : vérifions sa maille.

In [5]:
temp = pd.read_csv(fao_dir / "temp.csv")
par_pays_annee = temp.groupby(["country", "year"]).size()
print(par_pays_annee.value_counts().sort_index().head(8))
print("maximum :", par_pays_annee.max(), "lignes pour un même pays et une même année")

1    20520
2     3451
3     1192
4      197
5      791
6      742
7      191
8      190
Name: count, dtype: int64
maximum : 52 lignes pour un même pays et une même année


La plupart des couples (pays, année) ont une ligne, mais certains en ont 2, 3, 5… jusqu'à des dizaines.

### ✍️ À toi de jouer

Trouve les 5 pays qui ont le plus de lignes par année dans `temp.csv`, puis propose une façon d'obtenir **une seule** température par pays et par année.

In [6]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
print(par_pays_annee.groupby("country").max().sort_values(ascending=False).head(5))
temp_unique = temp.groupby(["country", "year"], as_index=False)["avg_temp"].mean()
print(temp_unique.duplicated(["country", "year"]).sum())  # 0 : la clé est unique
```

**Pourquoi ?** on agrège **avant** de joindre : une moyenne des stations donne une valeur par (pays, année). On aurait pu choisir la médiane ; l'important est de décider et de l'écrire. Joindre d'abord puis dédoublonner après coup ne marche pas : les lignes n'ont pas la même température, ce ne sont pas des doublons exacts.
</details>

## ✅ À retenir

1. Première question sur une table : **une ligne = quoi ?** Écris-le.
2. `df.duplicated(subset=cle).sum()` doit valoir 0 pour une clé ; sinon on cherche pourquoi.
3. Une jointure multiplie les lignes quand la table de droite a plusieurs lignes par clé.
4. Les doublons faussent les moyennes (un pays pèse trop) et les évaluations (mêmes lignes en train et en test).
5. On agrège la table de droite **avant** la jointure, et on contrôle la clé **après**.